[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/xpbd/xpbd.ipynb)

# Extended Position-Based Dynamics of Rigid Links

This example simulates a chain of rigid triaxial ellipsoidal links hanging from a fixed pin and released from a horizontal arc under gravity. Because the links are extended rigid bodies with three unequal principal moments of inertia, their whip-like swing couples translation with axial twisting and tumbling. Extended position-based dynamics advances every link freely at each substep and then projects its spherical joints closed by nudging separated anchors back along the line between them in proportion to each body's rotational and translational compliance.

Each link pose is a motor and its body-frame motion rate is a bivector, while its inertia maps `Rate` to `Forque` ([Gunn, 2011](https://arxiv.org/pdf/1101.4542); [Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)). Joining two world anchors `world_anchors[0] & world_anchors[1]` forms the constraint line directly: its norm is the separation gap and its normalized direction is the unit forque along which the correcting impulse acts. Pulling that unit forque into each body frame via `motors << unit_forque`, mapping through `inverse_inertias`, and pairing `twists & unit_forque` gives both bodies' effective compliances along the line without splitting linear and angular parts.

In the matrix Newton-Euler notation of [Macklin et al., 2016](https://matthias-research.github.io/pages/publications/XPBD.pdf), [Müller et al., 2020](https://matthias-research.github.io/pages/publications/PBDBodies.pdf) and [Macklin et al., 2019](https://mmacklin.com/smallsteps.pdf), closing a spherical joint between anchor offsets $\mathbf r_a, \mathbf r_b$ builds the separation $\Delta\mathbf x = (\mathbf x_b + \mathbf R_b\mathbf r_b) - (\mathbf x_a + \mathbf R_a\mathbf r_a)$ with direction $\mathbf n = \Delta\mathbf x / \|\Delta\mathbf x\|$, assembles the generalized inverse mass $w_a + w_b = m_a^{-1} + m_b^{-1} + (\mathbf r_a \times \mathbf R_a^\top\mathbf n)^\top\mathbf I_a^{-1}(\mathbf r_a \times \mathbf R_a^\top\mathbf n) + (\mathbf r_b \times \mathbf R_b^\top\mathbf n)^\top\mathbf I_b^{-1}(\mathbf r_b \times \mathbf R_b^\top\mathbf n)$, solves $\Delta\lambda = -\|\Delta\mathbf x\| / (w_a + w_b + \alpha / \Delta t^2)$, and applies separate centre-of-mass shifts $\pm m^{-1}\Delta\lambda\,\mathbf n$ alongside quaternion updates from $\mathbf I^{-1}(\mathbf r \times \mathbf R^\top\mathbf n)\Delta\lambda$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from examples.mechanics import lie_integrators as lie
from examples.mechanics.xpbd import render, scenarios
from examples.util.animation import show_states

context = NumpyContext(ga)
mv = context.multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
Point = ga.gatype.antivector()                                                  # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Rate = ga.gatype.bivector()                                                     # Bivector+
Forque = ga.gatype.antibivector()                                               # Antibivector-
Inertia = ga.gatype((Forque, Rate))                                             # Forque- <- Rate+
InverseInertia = ga.gatype((Rate, Forque))                                      # Rate+ <- Forque-
origin = mv.w.dual()                                                            # [] Point-

PairPartition = tuple[np.ndarray, Point]

## Curved chain and red-black joint partitions

Six axial unit masses per link reproduce the principal moments of a triaxial solid ellipsoid, while body zero carries zero inverse inertia so it remains pinned at the origin. Repeating one neighbour motor places the links along a horizontal arc, and splitting even and odd joints forms two disjoint partitions whose body pairs relax simultaneously in each batch.

In [ ]:
motors, rates, first_moments, inertias, inverse_inertias, gravity, partitions = scenarios.curved_chain(
    context, scenarios.n_links, scenarios.spacing, scenarios.semi_axes, scenarios.curl, scenarios.gravity_acceleration,
)
compliance = scenarios.compliance
damping = scenarios.damping

## Position and velocity constraint projections

Joining a joint's two world anchors yields the separation line directly, and joining each anchor with its body-frame velocity `anchors.regressive(anchors.commutator(rates))` yields its world velocity line when pushed forward by `motors`. Both projections pull the unit line back into each body frame, evaluate `twists & unit_forque` for the directional compliance, and apply equal and opposite impulses.

In [ ]:
def position_projection(
    motors: Motor, anchors: Point, inverse_inertias: InverseInertia, compliance: float, dt: float,
) -> Motor:
    """Distance-constraint projection moves each body pair (`[2, ...] Rotor+`) along the line between its anchors."""
    world_anchors = motors >> anchors                                           # [2, ...] Point-
    # Joining the two world anchors gives the separation as its norm and the impulse line as its normalized direction;
    # the tiny offset keeps closed joints at zero impulse without branching.
    line = world_anchors[0] & world_anchors[1]                                  # [...] Forque-
    separation = line.norm()                                                    # [...] Scalar+
    unit_forque = motors << (line / (separation + 1e-24))                       # [2, ...] Forque-
    # Mapping the unit forque through each body's inverse inertia yields its twist response, and pairing back gives the
    # effective inverse mass along the line.
    twists = inverse_inertias(unit_forque)                                      # [2, ...] Rate+
    total_compliance = (twists & unit_forque).sum(axis=0) + compliance / dt**2 + 1e-24   # [...] Scalar+
    impulse = twists * (separation / total_compliance * np.array([[1], [-1]]))  # [2, ...] Rate+
    return motors * (impulse * -0.5).exp()


def velocity_projection(
    motors: Motor, rates: Rate, anchors: Point, inverse_inertias: InverseInertia,
) -> Rate:
    """Velocity-constraint projection cancels relative anchor motion across each body pair (`[2, ...] Rate+`)."""
    # Joining each anchor with its body-frame velocity produces its world velocity line when pushed forward by the motor.
    anchor_velocities = motors >> anchors.regressive(anchors.commutator(rates))   # [2, ...] Forque-
    relative_velocity = anchor_velocities[1] - anchor_velocities[0]             # [...] Forque-
    speed = relative_velocity.scalar_norm_squared().square_root()               # [...] Scalar+
    unit_forque = motors << (relative_velocity / (speed + 1e-24))               # [2, ...] Forque-
    twists = inverse_inertias(unit_forque)                                      # [2, ...] Rate+
    total_compliance = (twists & unit_forque).sum(axis=0) + 1e-24               # [...] Scalar+
    impulse = twists * (speed / total_compliance * np.array([[1], [-1]]))       # [2, ...] Rate+
    return rates + impulse

## Substepping and swinging motion

Each substep advances the links freely with a fourth-order Lie Runge-Kutta step under gravity and Rayleigh damping, relaxes both joint partitions in Gauss-Seidel order, recovers body-frame rates from the finite motor change, and cancels relative anchor velocities. Four substeps per frame keep joint separations within a few millimetres, hidden inside the overlapping ellipsoid tips.

Ray-tracing and streaming the 151 frames on the GPU takes a few seconds.

In [ ]:
n_steps = scenarios.n_steps
n_substeps = scenarios.n_substeps
dt = scenarios.dt
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def substep(
    motors: Motor,
    rates: Rate,
    first_moments: Point,
    inertias: Inertia,
    inverse_inertias: InverseInertia,
    gravity: Point,
    compliance: float,
    damping: float,
    partitions: list[PairPartition],
    dt: float,
) -> tuple[Motor, Rate]:
    """One XPBD substep advances the body poses (`[...] Rotor+`) and body-frame rates (`[...] Rate+`) by `dt`."""
    def external_forque(motor: Motor, rate: Rate) -> Forque:
        weight = first_moments & (motor << gravity)                             # [...] Forque-
        return weight - inertias(rate) * damping

    relaxed_motors, _ = lie.explicit_rk4(motors, rates, inertias, inverse_inertias, dt, external_forque)
    for pairs, anchors in partitions:
        projected = position_projection(
            relaxed_motors[pairs], anchors, inverse_inertias[pairs], compliance, dt,
        )
        relaxed_motors = relaxed_motors.at[pairs].set(projected)

    # Recovering body rates from the finite motor displacement keeps velocities consistent with the projected poses.
    next_motors = relaxed_motors.normalized().retyped(Motor)                    # [...] Rotor+
    next_rates = ((~motors * next_motors).log() * (-2 / dt)).retyped(Rate)      # [...] Rate+
    for pairs, anchors in partitions:
        resolved = velocity_projection(
            next_motors[pairs], next_rates[pairs], anchors, inverse_inertias[pairs],
        )
        next_rates = next_rates.at[pairs].set(resolved).retyped(Rate)
    return next_motors, next_rates


def advance(
    motors: Motor,
    rates: Rate,
    first_moments: Point,
    inertias: Inertia,
    inverse_inertias: InverseInertia,
    gravity: Point,
    compliance: float,
    damping: float,
    partitions: list[PairPartition],
    n_substeps: int,
    dt: float,
) -> tuple[Motor, Rate]:
    """`n_substeps` equal XPBD substeps advance the body poses and rates across one frame interval `dt`."""
    for _ in range(n_substeps):
        motors, rates = substep(
            motors, rates, first_moments, inertias, inverse_inertias,
            gravity, compliance, damping, partitions, dt / n_substeps,
        )
    return motors, rates

In [ ]:
def swing_motors(
    motors: Motor,
    rates: Rate,
    first_moments: Point,
    inertias: Inertia,
    inverse_inertias: InverseInertia,
    gravity: Point,
    compliance: float,
    damping: float,
    partitions: list[PairPartition],
    n_steps: int,
    n_substeps: int,
    dt: float,
) -> Iterator[Motor]:
    """Yields the body poses (`[n_bodies] Rotor+`) initially and after each of `n_steps` frame intervals."""
    yield motors
    for _ in range(n_steps):
        motors, rates = advance(
            motors, rates, first_moments, inertias, inverse_inertias,
            gravity, compliance, damping, partitions, n_substeps, dt,
        )
        yield motors


def joint_separations(motors: Motor, partitions: list[PairPartition]) -> Scalar:
    """Euclidean anchor separations (`[...] Scalar+`) across all partitions come from the norm of the joined anchors."""
    def partition_separations(partition: PairPartition) -> Scalar:
        pairs, anchors = partition
        world_anchors = motors[..., pairs] >> anchors                           # [..., 2, ...] Point-
        return (world_anchors[..., 0, :] & world_anchors[..., 1, :]).norm()

    return concatenate([partition_separations(partition) for partition in partitions], axis=-1)


recorded_motors = show_states(
    swing_motors(
        motors, rates, first_moments, inertias, inverse_inertias,
        gravity, compliance, damping, partitions, n_steps, n_substeps, dt,
    ),
    render.animate_chain,
    scenarios.semi_axes,
    scenarios.spacing,
    name="xpbd_chain",
    duration_ms=frame_duration_ms,
    n_frames=n_steps + 1,
);
motor_history = stack(recorded_motors)                                          # [n_steps + 1, ...] Rotor+

Alternating orange and teal links hang from the dark grey spherical pin at the upper left, while darker shading on alternate body-frame octants reveals each ellipsoid rolling about its long axis as the whip wave travels down and reflects from the free tip.

In [ ]:
# checks
separations = joint_separations(motor_history, partitions)                      # [n_steps + 1, ...] Scalar+
pin_drift = ((motor_history[:, 0] >> origin) & origin).norm()                   # [n_steps + 1] Scalar+
assert (separations < 0.1 * scenarios.spacing).all()
assert (pin_drift < 1e-6).all()